# FE BAR Bronze Silver AutoLoader Pipeline — Code & Build Evidence

**Pipeline:** FE BAR Bronze Silver AutoLoader Pipeline | **ID:** `c0db28b8-dbd2-432d-99cf-4ea2aea60274`  
**Catalog:** `workshop_guy_catalog` | **Schema:** `fe_bar_a0`  
**Workspace:** fevm-workshop-guy | **Date:** 2026-10-14  
**Branch:** `feature/bronze-silver-autoloader-pipeline`

This notebook contains the exact pipeline SQL code committed to the repo, followed by verification queries proving the pipeline ran successfully. Run the SQL cells to regenerate evidence outputs.

## Bronze Layer — Streaming Tables via AutoLoader

Four streaming tables ingest raw JSON from UC Volumes via `read_files` (AutoLoader).

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE dim_customer_bronze
COMMENT "Bronze layer: raw customer dimension data ingested via AutoLoader from UC Volume"
AS SELECT *
FROM STREAM read_files(
  '/Volumes/workshop_guy_catalog/fe_bar_a0/staging/customers/',
  format => 'json'
);

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE dim_merchant_bronze
COMMENT "Bronze layer: raw merchant dimension data ingested via AutoLoader from UC Volume"
AS SELECT *
FROM STREAM read_files(
  '/Volumes/workshop_guy_catalog/fe_bar_a0/staging/merchants/',
  format => 'json'
);

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE fact_transactions_bronze
COMMENT "Bronze layer: raw transaction fact data ingested via AutoLoader from UC Volume"
AS SELECT *
FROM STREAM read_files(
  '/Volumes/workshop_guy_catalog/fe_bar_a0/staging/transactions/',
  format => 'json'
);

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE fact_prior_fraud_cases_bronze
COMMENT "Bronze layer: raw prior fraud cases data ingested via AutoLoader from UC Volume"
AS SELECT *
FROM STREAM read_files(
  '/Volumes/workshop_guy_catalog/fe_bar_a0/staging/prior_fraud_cases/',
  format => 'json'
);

In [0]:
%sql
-- BRONZE LAYER: Row counts for all ingested tables
SELECT 'dim_customer_bronze' AS table_name, COUNT(*) AS row_count FROM workshop_guy_catalog.fe_bar_a0.dim_customer_bronze
UNION ALL
SELECT 'dim_merchant_bronze', COUNT(*) FROM workshop_guy_catalog.fe_bar_a0.dim_merchant_bronze
UNION ALL
SELECT 'fact_transactions_bronze', COUNT(*) FROM workshop_guy_catalog.fe_bar_a0.fact_transactions_bronze
UNION ALL
SELECT 'fact_prior_fraud_cases_bronze', COUNT(*) FROM workshop_guy_catalog.fe_bar_a0.fact_prior_fraud_cases_bronze
ORDER BY table_name;

## Silver Layer — Materialized Views with Dedup, Expectations & FGAC

Four materialized views apply deduplication (`ROW_NUMBER`), NULL-key expectations (`ON VIOLATION DROP ROW`), and fine-grained access control:
- **`dim_customer_silver`** — column mask on `avg_monthly_spend` (PII group only)
- **`fact_transactions_silver`** — row filter hiding `amount > 10000` from non-PII users

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW dim_customer_silver (
  customer_id STRING,
  home_region STRING,
  risk_segment STRING,
  tenure BIGINT,
  avg_monthly_spend DOUBLE MASK workshop_guy_catalog.fe_bar_a0.mask_avg_monthly_spend,
  CONSTRAINT valid_customer_id EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT "Silver layer: deduplicated customers with column mask on avg_monthly_spend (PII group sees all)"
AS
SELECT customer_id, home_region, risk_segment, tenure, avg_monthly_spend
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY customer_id) AS rn
  FROM dim_customer_bronze
) WHERE rn = 1;

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW dim_merchant_silver (
  CONSTRAINT valid_merchant_id EXPECT (merchant_id IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT "Silver layer: deduplicated merchants with NULL merchant_id rows dropped"
AS
SELECT merchant_id, country, mcc_category, risk_tier
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY merchant_id ORDER BY merchant_id) AS rn
  FROM dim_merchant_bronze
) WHERE rn = 1;

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW fact_transactions_silver (
  CONSTRAINT valid_txn_id EXPECT (txn_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_customer_id EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW
)
WITH ROW FILTER workshop_guy_catalog.fe_bar_a0.filter_high_amount_txns ON (amount)
COMMENT "Silver layer: deduplicated transactions; row filter hides amount > 10000 from non-PII users"
AS
SELECT txn_id, customer_id, merchant_id, amount, channel, device_id,
       geo_distance_from_home, is_fraud, `timestamp`
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY txn_id ORDER BY `timestamp` DESC) AS rn
  FROM fact_transactions_bronze
) WHERE rn = 1;

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW fact_prior_fraud_cases_silver (
  CONSTRAINT valid_case_id EXPECT (case_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_txn_id EXPECT (txn_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_customer_id EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT "Silver layer: deduplicated prior fraud cases with NULL business keys dropped"
AS
SELECT case_id, customer_id, txn_id, fraud_type, investigation_outcome,
       recovery_amount, resolution_date
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY case_id ORDER BY resolution_date DESC) AS rn
  FROM fact_prior_fraud_cases_bronze
) WHERE rn = 1;

In [0]:
%sql
-- SILVER LAYER: Row counts for deduplicated tables
SELECT 'dim_customer_silver' AS table_name, COUNT(*) AS row_count FROM workshop_guy_catalog.fe_bar_a0.dim_customer_silver
UNION ALL
SELECT 'dim_merchant_silver', COUNT(*) FROM workshop_guy_catalog.fe_bar_a0.dim_merchant_silver
UNION ALL
SELECT 'fact_transactions_silver', COUNT(*) FROM workshop_guy_catalog.fe_bar_a0.fact_transactions_silver
UNION ALL
SELECT 'fact_prior_fraud_cases_silver', COUNT(*) FROM workshop_guy_catalog.fe_bar_a0.fact_prior_fraud_cases_silver
ORDER BY table_name;

## Build Evidence — Quality Checks & FGAC Verification

The cells below verify the pipeline ran correctly: zero NULL business keys, zero duplicates, and FGAC policies active.

In [0]:
%sql
-- QUALITY CHECK: Verify zero NULLs in business key columns after expectations applied
SELECT 'dim_customer_silver' AS table_name, 'customer_id' AS key_col,
       SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_count
FROM workshop_guy_catalog.fe_bar_a0.dim_customer_silver
UNION ALL
SELECT 'dim_merchant_silver', 'merchant_id',
       SUM(CASE WHEN merchant_id IS NULL THEN 1 ELSE 0 END)
FROM workshop_guy_catalog.fe_bar_a0.dim_merchant_silver
UNION ALL
SELECT 'fact_transactions_silver', 'txn_id',
       SUM(CASE WHEN txn_id IS NULL THEN 1 ELSE 0 END)
FROM workshop_guy_catalog.fe_bar_a0.fact_transactions_silver
UNION ALL
SELECT 'fact_transactions_silver', 'customer_id',
       SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END)
FROM workshop_guy_catalog.fe_bar_a0.fact_transactions_silver
UNION ALL
SELECT 'fact_prior_fraud_cases_silver', 'case_id',
       SUM(CASE WHEN case_id IS NULL THEN 1 ELSE 0 END)
FROM workshop_guy_catalog.fe_bar_a0.fact_prior_fraud_cases_silver
ORDER BY table_name, key_col;

In [0]:
%sql
-- DEDUP CHECK: Verify no duplicate business keys in silver tables
SELECT 'dim_customer_silver' AS table_name, 'customer_id' AS key_col,
       COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_count
FROM workshop_guy_catalog.fe_bar_a0.dim_customer_silver
UNION ALL
SELECT 'dim_merchant_silver', 'merchant_id',
       COUNT(*) - COUNT(DISTINCT merchant_id)
FROM workshop_guy_catalog.fe_bar_a0.dim_merchant_silver
UNION ALL
SELECT 'fact_transactions_silver', 'txn_id',
       COUNT(*) - COUNT(DISTINCT txn_id)
FROM workshop_guy_catalog.fe_bar_a0.fact_transactions_silver
UNION ALL
SELECT 'fact_prior_fraud_cases_silver', 'case_id',
       COUNT(*) - COUNT(DISTINCT case_id)
FROM workshop_guy_catalog.fe_bar_a0.fact_prior_fraud_cases_silver
ORDER BY table_name;

In [0]:
%sql
-- COLUMN MASK: avg_monthly_spend is masked for non-PII users (shows NULL)
SELECT * FROM workshop_guy_catalog.fe_bar_a0.dim_customer_silver LIMIT 10;

In [0]:
%sql
-- ROW FILTER: Rows with amount > 10000 are hidden for non-PII users
SELECT MAX(amount) AS max_visible_amount,
       COUNT(*) AS visible_row_count,
       SUM(CASE WHEN amount > 10000 THEN 1 ELSE 0 END) AS rows_over_10k
FROM workshop_guy_catalog.fe_bar_a0.fact_transactions_silver;

In [0]:
%sql
DESCRIBE FUNCTION EXTENDED workshop_guy_catalog.fe_bar_a0.filter_high_amount_txns;

In [0]:
%sql
DESCRIBE FUNCTION EXTENDED workshop_guy_catalog.fe_bar_a0.mask_avg_monthly_spend;